# Tarea Individual – Análisis Geoespacial de Ventas
## Cadena de Tiendas de Comestibles – Región Metropolitana de Chile

**Caso de aplicación:** Una cadena de tiendas de comestibles desea analizar sus ventas y el comportamiento de los clientes en diferentes ubicaciones para mejorar su estrategia de marketing.

**Dataset:** `dataset_tarea_ind.xlsx` · 40 643 órdenes, período enero–marzo 2025  
**GeoJSON:** `comunas_metropolitana-1.geojson` · 37 comunas del Gran Santiago

---

## 0. Configuración del entorno

In [ ]:
!pip install folium geopandas branca openpyxl -q

### Carga de archivos desde Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─────────────────────────────────────────────────────────────
# AJUSTA esta ruta a la carpeta donde subiste los archivos
# en tu Google Drive. Por ejemplo:
#   /content/drive/MyDrive/tarea_geoespacial/
# ─────────────────────────────────────────────────────────────
from pathlib import Path

DRIVE_DIR    = Path('/content/drive/MyDrive/tarea_geoespacial')
RUTA_EXCEL   = DRIVE_DIR / 'dataset_tarea_ind.xlsx'
RUTA_GEOJSON = DRIVE_DIR / 'comunas_metropolitana-1.geojson'

# ── Verificación ──
print(f"Directorio configurado: {DRIVE_DIR}")
for ruta in [RUTA_EXCEL, RUTA_GEOJSON]:
    estado = "✓ Encontrado" if ruta.exists() else "✗ NO encontrado – revisa la ruta DRIVE_DIR"
    print(f"  {estado}: {ruta.name}")

### Librerías

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import geopandas as gpd
import folium
from folium.plugins import HeatMap

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 13, 'axes.labelsize': 11})
print("Librerías cargadas correctamente.")

---
## 1. Selección y justificación de variables geoespaciales

### 1.1 Carga y limpieza del dataset

In [ ]:
df_raw = pd.read_excel(RUTA_EXCEL)
print(f'Dimensiones originales: {df_raw.shape}')
df_raw.head(3)

In [ ]:
def parse_float(series):
    """Convierte columnas con coma decimal (formato chileno) a float."""
    return (
        series.astype(str)
              .str.strip()
              .str.replace(',', '.', regex=False)
              .astype(float)
    )

df = df_raw.copy()
for col in ['venta_neta', 'lat', 'lng', 'kms_dist', 'lat_cd', 'lng_cd']:
    df[col] = parse_float(df[col])

df['fecha_compra'] = pd.to_datetime(df['fecha_compra'], dayfirst=True)
df['mes']          = df['fecha_compra'].dt.month
df['semana']       = df['fecha_compra'].dt.isocalendar().week.astype(int)
df['state']        = df['state'].str.title()

print(f'Dimensiones finales  : {df.shape}')
print(f'Periodo              : {df["fecha_compra"].min().date()} -> {df["fecha_compra"].max().date()}')
print(f'Comunas unicas       : {df["comuna"].nunique()}')
print(f'Centros distribucion : {df["centro_dist"].nunique()}')
df.dtypes

### 1.2 Variables geoespaciales identificadas y justificación

| Variable | Tipo | Justificación |
|---|---|---|
| `lat` / `lng` | Punto de venta (coordenadas del cliente) | Permiten ubicar exactamente dónde se originó cada orden, habilitando análisis de densidad y hotspots a nivel de punto. |
| `comuna` | Polígono (unidad administrativa) | Agrega las ventas a una unidad territorial reconocible, compatible con el GeoJSON para mapas coropléticos. |
| `centro_dist` + `lat_cd` / `lng_cd` | Punto de infraestructura logística | Ubica los 13 centros de distribución; permite analizar la relación entre cobertura logística y volumen de ventas. |
| `kms_dist` | Distancia euclidiana cliente→CD | Variable geoespacial derivada: mide el alcance de cada centro y permite correlacionar distancia con venta neta o canal. |
| `city` | Provincia | Nivel de agregación intermedio entre región y comuna; útil para comparar provincias. |

> **Dimensión adicional:** Las coordenadas transforman un análisis tabular en uno territorial: ¿*dónde* vendemos más?, ¿qué zonas están subatendidas?, ¿es eficiente la ubicación de los CDs?

---
## 2. Visualización básica de datos geográficos

### 2.1 Distribución de ventas por canal y provincia

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

canal_venta = df.groupby('canal')['venta_neta'].sum().sort_values(ascending=False)
axes[0].bar(canal_venta.index, canal_venta.values / 1e6,
            color=['#2196F3', '#FF9800'], edgecolor='white', linewidth=1.2)
axes[0].set_title('Venta Total por Canal')
axes[0].set_ylabel('Venta Neta (millones CLP)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))
for bar, val in zip(axes[0].patches, canal_venta.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'${val/1e6:.1f}M', ha='center', va='bottom', fontsize=10)

city_cnt = df['city'].value_counts()
axes[1].bar(city_cnt.index, city_cnt.values,
            color=sns.color_palette('Set2', len(city_cnt)), edgecolor='white')
axes[1].set_title('Ordenes por Provincia')
axes[1].set_ylabel('Numero de ordenes')
axes[1].tick_params(axis='x', rotation=30)
for bar, val in zip(axes[1].patches, city_cnt.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{val:,}', ha='center', va='bottom', fontsize=9)

cd_venta = df.groupby('centro_dist')['venta_neta'].sum().sort_values()
axes[2].barh(cd_venta.index, cd_venta.values / 1e6,
             color=sns.color_palette('Blues_d', len(cd_venta)))
axes[2].set_title('Venta Total por Centro de Distribucion')
axes[2].set_xlabel('Venta Neta (millones CLP)')
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))

plt.tight_layout(pad=2)
plt.show()

**Análisis:** El canal *App* concentra la mayor parte de las órdenes, mientras que la provincia de Santiago domina en volumen. El Centro de Distribución 1 y 2 son los de mayor facturación.

### 2.2 Top 15 comunas por venta neta total

In [ ]:
top15 = (
    df.groupby('comuna')['venta_neta']
      .sum()
      .sort_values(ascending=False)
      .head(15)
)

fig, ax = plt.subplots(figsize=(11, 5))
palette = sns.color_palette('YlOrRd', 15)[::-1]
bars = ax.barh(top15.index[::-1], top15.values[::-1] / 1e6,
               color=palette, edgecolor='white')

for bar, val in zip(bars, top15.values[::-1]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            f'${val/1e6:.1f}M', va='center', fontsize=9)

ax.set_title('Top 15 Comunas por Venta Neta Total (ene-mar 2025)')
ax.set_xlabel('Venta Neta (millones CLP)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))
ax.set_xlim(0, top15.max() / 1e6 * 1.15)
plt.tight_layout()
plt.show()

**Análisis:** Las Condes, Maipú y Ñuñoa lideran la venta neta. El top 15 es geográficamente diverso: incluye comunas del oriente (Las Condes, Providencia), poniente (Maipú, Pudahuel) y sur (La Florida, Puente Alto).

### 2.3 Relación distancia al CD vs. venta neta

In [ ]:
muestra = df.sample(n=3000, random_state=42)
orden_cd = df.groupby('centro_dist')['kms_dist'].median().sort_values().index

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for canal, grp in muestra.groupby('canal'):
    axes[0].scatter(grp['kms_dist'], grp['venta_neta'] / 1e3,
                    alpha=0.35, s=18, label=canal)
z = np.polyfit(muestra['kms_dist'], muestra['venta_neta'] / 1e3, 1)
xline = np.linspace(muestra['kms_dist'].min(), muestra['kms_dist'].max(), 100)
axes[0].plot(xline, np.poly1d(z)(xline), 'k--', linewidth=1.2, label='Tendencia')
axes[0].set_xlabel('Distancia al Centro de Distribucion (km)')
axes[0].set_ylabel('Venta Neta (miles CLP)')
axes[0].set_title('Distancia al CD vs. Venta Neta por Canal')
axes[0].legend(title='Canal')

axes[1].boxplot(
    [df[df['centro_dist'] == cd]['kms_dist'].values for cd in orden_cd],
    labels=[cd.replace('Centro Distribucion ', 'CD ') for cd in orden_cd],
    patch_artist=True,
    boxprops=dict(facecolor='#90CAF9'),
    medianprops=dict(color='#D32F2F', linewidth=2),
    flierprops=dict(marker='o', markersize=3, alpha=0.3)
)
axes[1].set_title('Distribucion de Distancia por Centro de Distribucion')
axes[1].set_xlabel('Centro de Distribucion')
axes[1].set_ylabel('Distancia (km)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout(pad=2)
plt.show()

**Análisis:** No existe correlación negativa fuerte entre distancia y venta, lo que sugiere que la propuesta digital supera la barrera geográfica. CD 2 y CD 7 cubren las zonas más extensas.

### 2.4 Evolución temporal de ventas y órdenes

In [ ]:
daily = (
    df.groupby(['fecha_compra', 'canal'])
      .agg(venta=('venta_neta', 'sum'), ordenes=('orden', 'count'))
      .reset_index()
)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for canal, grp in daily.groupby('canal'):
    axes[0].plot(grp['fecha_compra'], grp['venta'] / 1e6,
                 marker='.', markersize=3, linewidth=1.2, label=canal)
    axes[1].plot(grp['fecha_compra'], grp['ordenes'],
                 marker='.', markersize=3, linewidth=1.2, label=canal)

axes[0].set_title('Venta Neta Diaria por Canal')
axes[0].set_ylabel('Venta Neta (millones CLP)')
axes[0].legend(title='Canal')
axes[1].set_title('Numero de Ordenes Diarias por Canal')
axes[1].set_ylabel('Ordenes')
axes[1].set_xlabel('Fecha')
axes[1].legend(title='Canal')

fig.autofmt_xdate(rotation=30)
plt.tight_layout()
plt.show()

---
## 3. Gráficos geoespaciales avanzados con Folium

### 3.1 Mapa de calor – Intensidad de ventas

In [ ]:
CENTRO_RM = [-33.47, -70.65]
cd_coords = df.groupby('centro_dist')[['lat_cd', 'lng_cd']].first().reset_index()

mapa_heat = folium.Map(location=CENTRO_RM, zoom_start=11, tiles='CartoDB positron')

heat_data = (
    df[['lat', 'lng', 'venta_neta']]
    .dropna()
    .assign(peso=lambda d: d['venta_neta'] / d['venta_neta'].max())
    [['lat', 'lng', 'peso']]
    .values.tolist()
)

HeatMap(
    heat_data,
    radius=12,
    blur=15,
    max_zoom=13,
    min_opacity=0.3,
    gradient={
        '0.2': '#1a237e',
        '0.4': '#1976D2',
        '0.6': '#43A047',
        '0.8': '#FDD835',
        '1.0': '#D32F2F'
    }
).add_to(mapa_heat)

for _, row in cd_coords.iterrows():
    folium.Marker(
        location=[row['lat_cd'], row['lng_cd']],
        popup=folium.Popup(row['centro_dist'], max_width=200),
        icon=folium.Icon(color='red', icon='home')
    ).add_to(mapa_heat)

leyenda = '''
<div style="position:fixed;bottom:40px;left:40px;z-index:1000;
     background:rgba(255,255,255,0.92);padding:10px 14px;
     border-radius:8px;border:1px solid #ccc;font-size:12px;">
  <b>Intensidad de Ventas</b><br>
  <span style="background:#1a237e;padding:2px 10px;"></span> Baja<br>
  <span style="background:#1976D2;padding:2px 10px;"></span> Media-baja<br>
  <span style="background:#43A047;padding:2px 10px;"></span> Media<br>
  <span style="background:#FDD835;padding:2px 10px;"></span> Alta<br>
  <span style="background:#D32F2F;padding:2px 10px;"></span> Muy alta<br>
  <hr style="margin:4px 0">
  <span style="color:red;">&#9679;</span> Centro de Distribucion
</div>
'''
mapa_heat.get_root().html.add_child(folium.Element(leyenda))
mapa_heat

**Por qué el HeatMap supera a los gráficos básicos:** Mientras un gráfico de barras muestra totales agregados por comuna, el mapa de calor revela la distribución espacial *continua* de las ventas: clusters, corredores de alta actividad y zonas vacías dentro de una misma comuna.

### 3.2 Mapa de burbujas – Venta neta por comuna

In [ ]:
resumen_comuna = (
    df.groupby('comuna')
      .agg(
          venta_total   = ('venta_neta', 'sum'),
          ordenes       = ('orden', 'count'),
          unidades_prom = ('unidades', 'mean'),
          lat_med       = ('lat', 'median'),
          lng_med       = ('lng', 'median')
      )
      .reset_index()
)

mapa_burbuja = folium.Map(location=CENTRO_RM, zoom_start=11, tiles='CartoDB positron')
venta_max    = resumen_comuna['venta_total'].max()

for _, row in resumen_comuna.iterrows():
    radio = (row['venta_total'] / venta_max) * 35 + 5
    color = '#E53935' if row['venta_total'] > resumen_comuna['venta_total'].quantile(0.75) else '#1E88E5'
    folium.CircleMarker(
        location=[row['lat_med'], row['lng_med']],
        radius=radio,
        color='white', weight=1,
        fill=True, fill_color=color, fill_opacity=0.65,
        popup=folium.Popup(
            f"<b>{row['comuna']}</b><br>"
            f"Venta total: ${row['venta_total']:,.0f}<br>"
            f"Ordenes: {row['ordenes']:,}<br>"
            f"Unidades prom/orden: {row['unidades_prom']:.1f}",
            max_width=220
        ),
        tooltip=row['comuna']
    ).add_to(mapa_burbuja)

mapa_burbuja

---
## 4. Patrones geográficos y Hotspots – Mapa Coroplético

### 4.1 Carga del GeoJSON y merge con ventas

In [ ]:
gdf = gpd.read_file(RUTA_GEOJSON)
gdf = gdf.rename(columns={'name': 'comuna'})

stats_comuna = (
    df.groupby('comuna')
      .agg(
          venta_total  = ('venta_neta', 'sum'),
          ordenes      = ('orden', 'count'),
          ticket_prom  = ('venta_neta', 'mean'),
          unidades_tot = ('unidades', 'sum'),
          pct_app      = ('canal', lambda x: (x == 'App').mean() * 100)
      )
      .reset_index()
)

gdf_merged = gdf.merge(stats_comuna, on='comuna', how='left')
print(f'Comunas con datos: {gdf_merged["venta_total"].notna().sum()} / {len(gdf_merged)}')
gdf_merged[['comuna', 'venta_total', 'ordenes', 'ticket_prom']].sort_values('venta_total', ascending=False).head(10)

### 4.2 Mapa coroplético interactivo

In [ ]:
mapa_choro = folium.Map(location=CENTRO_RM, zoom_start=11, tiles='CartoDB positron')

folium.Choropleth(
    geo_data=str(RUTA_GEOJSON),
    name='Venta Neta por Comuna',
    data=stats_comuna,
    columns=['comuna', 'venta_total'],
    key_on='feature.properties.name',
    fill_color='YlOrRd',
    fill_opacity=0.75,
    line_opacity=0.4,
    legend_name='Venta Neta Total (CLP)',
    nan_fill_color='#e0e0e0',
    highlight=True
).add_to(mapa_choro)

with open(RUTA_GEOJSON) as f:
    geojson_data = json.load(f)

lookup = stats_comuna.set_index('comuna').to_dict('index')
for feature in geojson_data['features']:
    nombre = feature['properties']['name']
    datos  = lookup.get(nombre, {})
    feature['properties']['venta_total'] = f"${datos.get('venta_total', 0):,.0f}"
    feature['properties']['ordenes']     = f"{datos.get('ordenes', 0):,}"
    feature['properties']['ticket_prom'] = f"${datos.get('ticket_prom', 0):,.0f}"
    feature['properties']['pct_app']     = f"{datos.get('pct_app', 0):.1f}%"

folium.GeoJson(
    geojson_data,
    style_function=lambda x: {'fillColor': 'transparent', 'color': 'transparent', 'weight': 0},
    tooltip=folium.GeoJsonTooltip(
        fields=['name', 'venta_total', 'ordenes', 'ticket_prom', 'pct_app'],
        aliases=['Comarca', 'Venta Total', 'Ordenes', 'Ticket Prom.', '% App'],
        localize=True, sticky=True, style='font-size:12px;'
    )
).add_to(mapa_choro)

for _, row in cd_coords.iterrows():
    venta_cd = df[df['centro_dist'] == row['centro_dist']]['venta_neta'].sum()
    folium.CircleMarker(
        location=[row['lat_cd'], row['lng_cd']],
        radius=7,
        color='#0D47A1', fill=True, fill_color='#1976D2', fill_opacity=0.9,
        popup=folium.Popup(f"{row['centro_dist']}<br>Venta: ${venta_cd:,.0f}", max_width=220),
        tooltip=row['centro_dist'].replace('Centro Distribucion', 'CD')
    ).add_to(mapa_choro)

folium.LayerControl().add_to(mapa_choro)
mapa_choro

### 4.3 Mapa coroplético estático (GeoPandas)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

gdf_merged.plot(
    column='venta_total', ax=axes[0], cmap='YlOrRd', legend=True,
    missing_kwds={'color': '#eeeeee'}, edgecolor='white', linewidth=0.5,
    legend_kwds={'shrink': 0.6, 'label': 'Venta Neta Total (CLP)'}
)
for _, row in gdf_merged.sort_values('venta_total', ascending=False).head(8).iterrows():
    c = row.geometry.centroid
    axes[0].annotate(row['comuna'], xy=(c.x, c.y), ha='center', va='center',
                     fontsize=6.5, color='#212121', fontweight='bold')
axes[0].scatter(cd_coords['lng_cd'], cd_coords['lat_cd'],
                color='#0D47A1', zorder=5, s=60, marker='*', label='CD')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].set_title('Mapa Coropletico - Venta Neta por Comuna')
axes[0].axis('off')

gdf_merged.plot(
    column='ticket_prom', ax=axes[1], cmap='Blues', legend=True,
    missing_kwds={'color': '#eeeeee'}, edgecolor='white', linewidth=0.5,
    legend_kwds={'shrink': 0.6, 'label': 'Ticket Promedio (CLP)'}
)
axes[1].set_title('Mapa Coropletico - Ticket Promedio por Comuna')
axes[1].axis('off')

plt.suptitle('Patrones Geograficos de Ventas - RM Chile (ene-mar 2025)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 4.4 Identificación de hotspots

In [ ]:
stats_comuna['cuartil'] = pd.qcut(
    stats_comuna['venta_total'], q=4,
    labels=['Q1 Bajo', 'Q2 Medio-bajo', 'Q3 Medio-alto', 'Q4 Hotspot']
)

print('=== Hotspots - Cuartil superior (Q4) ===')
hotspots = stats_comuna[stats_comuna['cuartil'] == 'Q4 Hotspot'].sort_values('venta_total', ascending=False)
print(hotspots[['comuna', 'venta_total', 'ordenes', 'ticket_prom', 'pct_app']].to_string(index=False))

print('\n=== Zonas de baja actividad (Q1) ===')
bajas = stats_comuna[stats_comuna['cuartil'] == 'Q1 Bajo'].sort_values('venta_total')
print(bajas[['comuna', 'venta_total', 'ordenes']].to_string(index=False))

### 4.5 Hipótesis sobre patrones observados

| Patrón observado | Hipótesis explicativa |
|---|---|
| **Las Condes, Providencia, Vitacura** – alto ticket promedio | Mayor poder adquisitivo; compra de productos premium en mayor cantidad. |
| **Maipú y La Florida** – alto volumen de órdenes | Alta densidad poblacional y penetración digital de estratos medios. |
| **Borde sur (Buin, Calera de Tango)** – baja actividad | Mayor distancia a CDs; menor cobertura logística y adopción digital. |
| **Poniente (Pudahuel, Cerro Navia)** – actividad media | Potencial de crecimiento: alta densidad pero baja venta per cápita. |

---
## 5. Dashboard Interactivo con Streamlit

El siguiente código genera el archivo `dashboard.py`. Para ejecutarlo:

```bash
# Instalar dependencias
pip install streamlit streamlit-folium folium geopandas openpyxl

# Ejecutar (desde la misma carpeta que los archivos de datos)
streamlit run dashboard.py
```

In [ ]:
dashboard_code = r'''
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import streamlit as st
import pandas as pd
import numpy as np
import json
import geopandas as gpd
import folium
from folium.plugins import HeatMap
from streamlit_folium import st_folium
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

st.set_page_config(page_title="Dashboard Ventas RM", page_icon="map", layout="wide")

NOTEBOOK_DIR = Path(__file__).parent
RUTA_EXCEL   = NOTEBOOK_DIR / "dataset_tarea_ind.xlsx"
RUTA_GEOJSON = NOTEBOOK_DIR / "comunas_metropolitana-1.geojson"

@st.cache_data
def cargar_datos():
    df = pd.read_excel(RUTA_EXCEL)
    def pf(s):
        return s.astype(str).str.strip().str.replace(",", ".", regex=False).astype(float)
    for col in ["venta_neta", "lat", "lng", "kms_dist", "lat_cd", "lng_cd"]:
        df[col] = pf(df[col])
    df["fecha_compra"] = pd.to_datetime(df["fecha_compra"], dayfirst=True)
    df["mes"]  = df["fecha_compra"].dt.month
    df["state"] = df["state"].str.title()
    return df

@st.cache_data
def cargar_geojson():
    with open(RUTA_GEOJSON) as f:
        return json.load(f)

df_full = cargar_datos()
geojson = cargar_geojson()
CENTRO_RM = [-33.47, -70.65]

st.sidebar.title("Filtros")
canales = st.sidebar.multiselect("Canal", df_full["canal"].unique().tolist(),
                                  default=df_full["canal"].unique().tolist())
comunas_sel = st.sidebar.multiselect("Comunas", sorted(df_full["comuna"].unique().tolist()),
                                      default=sorted(df_full["comuna"].unique().tolist()))
fecha_min = df_full["fecha_compra"].min().date()
fecha_max = df_full["fecha_compra"].max().date()
rango = st.sidebar.date_input("Periodo", value=(fecha_min, fecha_max),
                               min_value=fecha_min, max_value=fecha_max)
vr = st.sidebar.slider("Rango Venta Neta (CLP)",
                        int(df_full["venta_neta"].min()), int(df_full["venta_neta"].max()),
                        (int(df_full["venta_neta"].min()), int(df_full["venta_neta"].max())))

f_ini = pd.Timestamp(rango[0]) if len(rango) == 2 else pd.Timestamp(fecha_min)
f_fin = pd.Timestamp(rango[1]) if len(rango) == 2 else pd.Timestamp(fecha_max)

df = df_full[
    df_full["canal"].isin(canales) &
    df_full["comuna"].isin(comunas_sel) &
    df_full["fecha_compra"].between(f_ini, f_fin) &
    df_full["venta_neta"].between(vr[0], vr[1])
]

st.title("Dashboard de Ventas Geoespaciales - Region Metropolitana")
st.markdown(f"**{df.shape[0]:,} ordenes** filtradas | Periodo {f_ini.date()} al {f_fin.date()}")
st.divider()

c1,c2,c3,c4,c5 = st.columns(5)
c1.metric("Venta Total",      f"${df['venta_neta'].sum()/1e6:.1f} M")
c2.metric("Ordenes",          f"{df.shape[0]:,}")
c3.metric("Ticket Promedio",  f"${df['venta_neta'].mean():,.0f}")
c4.metric("Comunas activas",  df["comuna"].nunique())
c5.metric("Unidades totales", f"{df['unidades'].sum():,}")
st.divider()

tab1, tab2, tab3, tab4 = st.tabs(["Analisis General","Mapa de Calor","Mapa Coropletico","Tendencias"])

with tab1:
    col1, col2 = st.columns(2)
    with col1:
        st.subheader("Top 10 comunas por venta neta")
        top10 = df.groupby("comuna")["venta_neta"].sum().sort_values(ascending=False).head(10)
        fig, ax = plt.subplots(figsize=(6,4))
        ax.barh(top10.index[::-1], top10.values[::-1]/1e6,
                color=sns.color_palette("YlOrRd",10)[::-1])
        ax.set_xlabel("Venta Neta (MM CLP)")
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:.0f}M"))
        plt.tight_layout(); st.pyplot(fig); plt.close()
    with col2:
        st.subheader("Participacion por canal")
        cv = df.groupby("canal")["venta_neta"].sum()
        fig2, ax2 = plt.subplots(figsize=(5,4))
        ax2.pie(cv.values, labels=cv.index, autopct="%1.1f%%",
                colors=["#2196F3","#FF9800"],
                wedgeprops=dict(edgecolor="white", linewidth=1.5))
        plt.tight_layout(); st.pyplot(fig2); plt.close()
    st.subheader("Tabla resumen por comuna")
    tabla = (df.groupby("comuna").agg(
        Venta_Total=("venta_neta","sum"), Ordenes=("orden","count"),
        Ticket_Prom=("venta_neta","mean"), Unidades=("unidades","sum"))
        .sort_values("Venta_Total", ascending=False).reset_index())
    tabla["Venta_Total"] = tabla["Venta_Total"].apply(lambda x: f"${x:,.0f}")
    tabla["Ticket_Prom"] = tabla["Ticket_Prom"].apply(lambda x: f"${x:,.0f}")
    st.dataframe(tabla, use_container_width=True)

with tab2:
    st.subheader("Mapa de Calor - Intensidad de Ventas")
    tiles = st.radio("Estilo de mapa", ["CartoDB positron","CartoDB dark_matter","OpenStreetMap"], horizontal=True)
    m = folium.Map(location=CENTRO_RM, zoom_start=11, tiles=tiles)
    hd = df[["lat","lng","venta_neta"]].dropna().assign(
        peso=lambda d: d["venta_neta"]/df["venta_neta"].max())[["lat","lng","peso"]].values.tolist()
    HeatMap(hd, radius=12, blur=15, min_opacity=0.3,
            gradient={"0.2":"#1a237e","0.4":"#1976D2","0.6":"#43A047","0.8":"#FDD835","1.0":"#D32F2F"}).add_to(m)
    cdc = df.groupby("centro_dist")[["lat_cd","lng_cd"]].first().reset_index()
    for _, row in cdc.iterrows():
        v = df[df["centro_dist"]==row["centro_dist"]]["venta_neta"].sum()
        folium.Marker([row["lat_cd"],row["lng_cd"]],
            popup=f"{row['centro_dist']}<br>${v:,.0f}",
            icon=folium.Icon(color="red",icon="home")).add_to(m)
    st_folium(m, width=900, height=500)

with tab3:
    st.subheader("Mapa Coropletico - Ventas por Zona")
    metrica = st.selectbox("Metrica", ["Venta Total","Numero de Ordenes","Ticket Promedio","% Canal App"])
    col_map = {"Venta Total":"venta_total","Numero de Ordenes":"ordenes",
               "Ticket Promedio":"ticket_prom","% Canal App":"pct_app"}
    stats = df.groupby("comuna").agg(
        venta_total=("venta_neta","sum"), ordenes=("orden","count"),
        ticket_prom=("venta_neta","mean"),
        pct_app=("canal", lambda x: (x=="App").mean()*100)).reset_index()
    col_sel = col_map[metrica]
    mc = folium.Map(location=CENTRO_RM, zoom_start=11, tiles="CartoDB positron")
    folium.Choropleth(geo_data=geojson, data=stats, columns=["comuna", col_sel],
        key_on="feature.properties.name", fill_color="YlOrRd",
        fill_opacity=0.75, line_opacity=0.4, legend_name=metrica,
        nan_fill_color="#e0e0e0", highlight=True).add_to(mc)
    lookup = stats.set_index("comuna").to_dict("index")
    gj2 = json.loads(json.dumps(geojson))
    for feat in gj2["features"]:
        n = feat["properties"]["name"]
        d = lookup.get(n, {})
        feat["properties"]["venta_total"] = f"${d.get('venta_total',0):,.0f}"
        feat["properties"]["ordenes"]     = f"{d.get('ordenes',0):,}"
        feat["properties"]["ticket_prom"] = f"${d.get('ticket_prom',0):,.0f}"
        feat["properties"]["pct_app"]     = f"{d.get('pct_app',0):.1f}%"
    folium.GeoJson(gj2,
        style_function=lambda x: {"fillColor":"transparent","color":"transparent","weight":0},
        tooltip=folium.GeoJsonTooltip(
            fields=["name","venta_total","ordenes","ticket_prom","pct_app"],
            aliases=["Comuna","Venta","Ordenes","Ticket","% App"],
            sticky=True, style="font-size:12px")).add_to(mc)
    st_folium(mc, width=900, height=500)
    st.dataframe(stats.sort_values(col_sel, ascending=False)[["comuna",col_sel]].reset_index(drop=True),
                 use_container_width=True)

with tab4:
    st.subheader("Evolucion Temporal de Ventas")
    agg = st.radio("Granularidad", ["Diario","Semanal"], horizontal=True)
    if agg == "Diario":
        tmp = df.groupby(["fecha_compra","canal"]).agg(
            venta=("venta_neta","sum"), ordenes=("orden","count")).reset_index()
        xcol = "fecha_compra"
    else:
        df2 = df.copy()
        df2["sem"] = df2["fecha_compra"] - pd.to_timedelta(df2["fecha_compra"].dt.dayofweek, unit="D")
        tmp = df2.groupby(["sem","canal"]).agg(
            venta=("venta_neta","sum"), ordenes=("orden","count")).reset_index()
        xcol = "sem"
    fig3, ax3 = plt.subplots(2,1,figsize=(11,6),sharex=True)
    for canal, grp in tmp.groupby("canal"):
        ax3[0].plot(grp[xcol], grp["venta"]/1e6, marker=".", markersize=3, linewidth=1.2, label=canal)
        ax3[1].plot(grp[xcol], grp["ordenes"],   marker=".", markersize=3, linewidth=1.2, label=canal)
    ax3[0].set_ylabel("Venta Neta (MM CLP)"); ax3[0].legend()
    ax3[1].set_ylabel("Ordenes"); ax3[1].legend()
    fig3.autofmt_xdate(rotation=30)
    plt.tight_layout(); st.pyplot(fig3); plt.close()

st.divider()
st.caption("Dashboard desarrollado con Streamlit | Datos: Cadena de Comestibles RM (ene-mar 2025)")
'''

with open('dashboard.py', 'w', encoding='utf-8') as f:
    f.write(dashboard_code.strip())

print("dashboard.py generado correctamente.")
print("Para ejecutar: streamlit run dashboard.py")

---
## Resumen ejecutivo

El análisis geoespacial de las 40 643 órdenes revela:

1. **Concentración geográfica:** Las 10 comunas líderes concentran ~60% de la facturación, combinando densidad poblacional (poniente/sur) con poder adquisitivo (oriente).
2. **Canal App dominante:** En la mayoría de comunas, App supera a Sitio en órdenes, orientando el marketing hacia optimización móvil.
3. **Brecha logística:** El sur-poniente (Maipú, Pudahuel) muestra alta demanda pero mayor distancia a CDs — oportunidad de optimización.
4. **Zonas de oportunidad:** Comunas del borde sur (Buin, Calera de Tango) presentan baja actividad relativa a su población.